# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a Croissant-formatted dataset using the `mlcroissant` library. It follows best practices for referencing dataset entities by their `@id` and demonstrates common data processing and visualization workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` (for example: record sets, fields, and columns).

In [ ]:
# List available RecordSets, Fields, and Columns by their @id
record_sets = list(dataset.record_sets)

print("Available record sets by @id:")
for rs in record_sets:
    print("- RecordSet @id:", rs['@id'], "; Name:", rs.get('name', 'N/A'))
    print("  Fields (@id):")
    for f in rs.get('field', []):
        if isinstance(f, dict):
            print("    -", f.get('@id'), ":", f.get('name', 'N/A'))
        else:
            print("    -", f)
    print("  Columns (@id):")
    for c in rs.get('column', []):
        if isinstance(c, dict):
            print("    -", c.get('@id'), ":", c.get('name', 'N/A'))
        else:
            print("    -", c)
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

*For this dataset, we select the principal tabular RecordSet containing the survey data, typically named or described as such in the RecordSet `@id` and metadata. Adjust the record_set_id as needed based on overview above.*

In [ ]:
# Example: Use the first available tabular record set for analysis
if len(record_sets) == 0:
    raise RuntimeError('No record sets found in the dataset')

# Choose the main survey RecordSet by inspecting `@id` or use first one
record_set_id = record_sets[0]['@id']
print(f"Selected record set @id for extraction: {record_set_id}")

# If there are multiple record sets, you can extract their @ids here
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

df = dataframes[record_set_id]
print(f"DataFrame columns for record set {record_set_id} (@id):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering, normalization, and grouping—on fields referenced by their `@id`. For demonstration, we pick a numeric field (for example: GAD-7 score or age).

> **Please adjust the `numeric_field_id` and `group_field_id` to match available fields (using the exact field `@id`).**

In [ ]:
# Choose fields by @id as identified above. Example placeholder ids; please adjust as needed.
possible_numeric_fields = [c for c in df.columns if c.lower().find('gad7') != -1 or c.lower().find('phq9') != -1 or c.lower().find('psq') != -1 or c.lower().find('age') != -1]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns)>0 else df.columns[0]
print(f"Using numeric field @id: {numeric_field_id}")

# Pick a group field (e.g., gender, education level). Adjust as per your data.
possible_group_fields = [c for c in df.columns if c.lower() in ['gender', 'sex', 'level_of_education', 'village', 'marital_status'] or c.lower().endswith('gender') or c.lower().endswith('education')]
group_field_id = possible_group_fields[0] if possible_group_fields else None
if group_field_id:
    print(f"Using group field @id: {group_field_id}")

# Filter, normalize and group
if df[numeric_field_id].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df)
else:
    print(f"Selected numeric_field_id '{numeric_field_id}' is not recognized as a numeric dtype. Please update field IDs.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: Distribution of selected score or breakdown by group.

> *Ensure Matplotlib and/or Seaborn are installed for visualization.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in df.columns and df[group_field_id].nunique() < 10:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print(f"Field '{numeric_field_id}' not available for plotting.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. You have:
- Loaded metadata and explored the Croissant schema.
- Identified record sets, fields, and columns by their `@id`.
- Extracted tabular survey data using record set `@id`.
- Processed and visualized numeric fields such as psychological assessment scores (for example, GAD-7, PHQ-9, or Age).
- Applied basic EDA operations and visualizations by referencing fields and groupings by their `@id`.

This workflow ensures transparency and reproducibility with FAIR AI-ready standards.